# 03: Model Training

Let's build an neral network for this

In [8]:
import json
import os

import numpy as np
import pandas as pd

In [9]:
# Run this notebook from the `ai/` directory.
HERE = os.getcwd()
DATA_DIR = os.path.join(HERE, "data")
IN_PATH = os.path.join(DATA_DIR, "pre_process_transaction_ds.csv")
TRAINED_WEIGHTS = os.path.join(DATA_DIR, "trained_model_weights.json")

In [10]:
FEATURE_NAMES = [
    "txVolume",            # tanh(log1p(avg value moved in ETH) / 3)
    "txFrequency",         # tanh(tx count / 50)
    "accountAge",          # tanh(active days / 365)
    "networkDegree",       # tanh(unique counterparties / 20)
    "timeRegularity",      # tanh(log1p(avg minutes between txs) / 6)
    "valueSentRatio",      # sent value / (sent + received)  -> 0..1, 0.5 neutral
    "inOutRatio",          # received txns / total txns      -> 0..1
    "degreeConcentration", # unique counterparties / tx count -> 0..1
    "valueVolatility",     # tanh(log1p(max value moved) / 3)
]

In [11]:
df = pd.read_csv(IN_PATH)
X = df[FEATURE_NAMES].to_numpy()
y = df["FLAG"].to_numpy()
print(f"Loaded {len(df):,} samples from {IN_PATH}")
print(f"Label balance: fraud={int(y.sum()):,} legit={int((1 - y).sum()):,}")

Loaded 9,841 samples from /home/ayush/Desktop/code/SecureTransac/ai/data/pre_process_transaction_ds.csv
Label balance: fraud=2,179 legit=7,662


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_class_weight

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


def balanced_sample_weights(y):
    classes = np.unique(y)
    w = compute_class_weight("balanced", classes=classes, y=y)
    weights = np.ones(len(y), dtype=float)
    for c, cw in zip(classes, w):
        weights[y == c] = cw
    return weights

print(f"Train: {len(X_train)}  Test: {len(X_test)}")

Train: 7872  Test: 1969


In [13]:
model = MLPClassifier(
    hidden_layer_sizes=(12, 6),
    activation="relu",
    solver="adam",
    max_iter=1000,
    random_state=42,
)
model.fit(X_train, y_train, sample_weight=balanced_sample_weights(y_train))
print(f"Training loss: {model.loss_:.4f} after {model.n_iter_} iterations")

Training loss: 0.2294 after 196 iterations


In [14]:
def evaluate(y_true, y_pred_prob, name, threshold=0.5):
    y_pred = (y_pred_prob >= threshold).astype(int)
    return {
        "model": name,
        "auc": round(float(roc_auc_score(y_true, y_pred_prob)), 4),
        "f1": round(float(f1_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred)), 4),
        "recall": round(float(recall_score(y_true, y_pred)), 4),
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    }

lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_train, y_train)
priors = np.full_like(y_test, float(np.mean(y_train)), dtype=float)

results = pd.DataFrame([
    evaluate(y_test, model.predict_proba(X_test)[:, 1], "MLP (real model)"),
    evaluate(y_test, lr.predict_proba(X_test)[:, 1], "LogisticRegression"),
    evaluate(y_test, priors, "Majority prior"),
])
results.set_index("model", inplace=True)
results

/home/ayush/Desktop/code/SecureTransac/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,auc,f1,precision,recall,accuracy
model,,,,,
MLP (real model),0.9606,0.7787,0.6840,0.9037,0.8862
LogisticRegression,0.9404,0.7160,0.6054,0.8761,0.8461
Majority prior,0.5000,0.0000,0.0000,0.0000,0.7786


In [15]:
cm = confusion_matrix(y_test, (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int))
print("Confusion matrix (fraud=1):")
print("  [[TN={} FP={}]\n   [FN={} TP={}]]".format(cm[0][0], cm[0][1], cm[1][0], cm[1][1]))

Confusion matrix (fraud=1):
  [[TN=1351 FP=182]
   [FN=42 TP=394]]


In [16]:
cv_aucs = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for tr_idx, va_idx in skf.split(X, y):
    m = MLPClassifier(hidden_layer_sizes=(12, 6), activation="relu", solver="adam",
                      max_iter=1000, random_state=42)
    m.fit(X[tr_idx], y[tr_idx], sample_weight=balanced_sample_weights(y[tr_idx]))
    cv_aucs.append(roc_auc_score(y[va_idx], m.predict_proba(X[va_idx])[:, 1]))

print(f"MLP 5-fold CV AUC: {np.mean(cv_aucs):.4f} (+/- {np.std(cv_aucs):.4f})")

MLP 5-fold CV AUC: 0.9686 (+/- 0.0034)


In [17]:
# Export the trained weights (notebook 04 copies them to the backend locations).
last = model.coefs_[-1]
assert last.shape[1] == 1, f"Unexpected binary output shape {last.shape}"

weights = {
    "layers": [],
}
for i in range(len(model.coefs_)):
    weights["layers"].append({
        "weights": model.coefs_[i].tolist(),
        "biases": model.intercepts_[i].tolist(),
        "activation": "relu" if i < len(model.coefs_) - 1 else "sigmoid",
    })

with open(TRAINED_WEIGHTS, "w") as f:
    json.dump(weights, f)
print(f"Exported {TRAINED_WEIGHTS} ({os.path.getsize(TRAINED_WEIGHTS):,} bytes, {len(weights['layers'])} layers)")

Exported /home/ayush/Desktop/code/SecureTransac/ai/data/trained_model_weights.json (4,521 bytes, 3 layers)
